# Forecasting

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [2]:
OUTPUTS = PROJECT_ROOT / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)

In [ ]:
from src.experiments.artifacts import ReconstructedStore

store = ReconstructedStore(OUTPUTS / "reconstructed")

In [3]:
import pandas as pd
import numpy as np

Repeat code from `imputation.ipynb`.

In [6]:
from src.data import load_ett, chronological_split, scale

train, test = chronological_split(load_ett())
train_s, test_s, scaler = scale(train=train, test=test)

Set horizon length.

In [2]:
H = 96

In [ ]:
truth = test_s["OT"].iloc[:H].to_numpy()

### Baseline Forecast with Original Complete Data

Exponential Smoothing

In [ ]:
from src.forecasters.exp_smoothing import ExpSmoothingForecaster

exp_base = ExpSmoothingForecaster()
exp_base.fit(train_s)
exp_base_forecast = exp_base.predict(horizon=H)

XGBoost

In [ ]:
from src.forecasters.xgb import XGBoostForecaster

xgb_base = XGBoostForecaster()
xgb_base_trained = xgb_base.fit(train_s)
xgb_base_forecast = xgb_base_trained.predict(horizon=H)

### Forecasting Experiment

In [ ]:
from src.experiments.forecasting import ForecastingExperiment
from src.forecasters import FORECASTERS

baselines = {
    "exp_smoothing": exp_base_forecast,
    "xgboost": xgb_base_forecast,
}

experiment = ForecastingExperiment(forecasters=list(FORECASTERS))
bv_df = experiment.run(horizon=H, store=store, truth=truth, baselines=baselines)

bv_df.to_parquet(OUTPUTS / "bias_variance.parquet")
bv_df